##### GFM Flood Day Count Workflow
Computes "number of days flooded" rasters from Copernicus Global Flood Monitoring (GFM)
- Served through the EODC STAC API
- Input variable searched: ensemble flood extent (model decision for flood)
- Output: Cloud-Optimised GeoTIFF per month at 100m resolution, DEFLATE compression and nodata = 255 (uint8)
    <!-- - All monthly share a identical spatial grid (shape, transform, CRS) for direct stacking. -->

Some design choices
- `odc-stac groupby="solar_day"`: deduplicates same-day satellite passes, eventually (not available in `stackstac`)
- `resolution` + `resampling`: load data at 100m (WorldPop match) with "max" aggregation
    - *any* 20m flooded pixel promotes to parent 100m cell
- `dask lazy loading` + `chunked computation`: keeps RAM under control even for large countries (hundreds of STAC items per month)
- `Month-by-month loop`: outputs one GeoTIFF per month, ready for downstream population-exposure analysis
<!-- - Country mask rasterized once:
    - guarantees identical output grid across months
    -  enables accurate QC stats (diff. clipped-nodata from satellite-no-coverage-nodata). -->

Requirements (see `requirements.txt`)

`odc-stac`, `pystac-client`, `dask[distributed]`, `rioxarray`, `rasterio`

In [ ]:
# 0.  IMPORTS

import logging
from datetime import date, datetime, timedelta
import importlib.metadata as lib_metadata
import json
import math
from pathlib import Path
import platform
import time

import geopandas as gpd
import xarray as xr
from dateutil.relativedelta import relativedelta
from dask.distributed import Client as DaskClient
import numpy as np
import odc.stac as odc_stac
import rioxarray    # noqa: F401 – registers the .rio accessor
import pycountry
from pystac_client import Client as STACClient
from rasterio.features import rasterize


In [ ]:
# set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


##### 1.  CONFIGURATION  — edit this section for your run

In [ ]:
# -- Country for consultation --
country_list = [
    # "Kenya",
    # "Mali",
    # "Benin",
    "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = country_list[0]

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

# -- Boundary file for consultation --
# NOTE: output is raster, admin level used only for geojson id
admin_level = "3"

# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path("./data_in/shapes/").iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read geojson into single country geometry gdf
gdf = gpd.read_file(
    "./data_in/shapes/" + f"{geojson_file[0]}"
).to_crs('EPSG:4326').dissolve()


In [ ]:
# -- Time window for consultation --
# NOTE: select last year or last 12 months of data
time_filter = "last_year" # "last_12_months" #

# Uses last updated Friday like Acled Updates (assumes fresh data on Wednesdays)
today = date.today()
weekday = today.weekday()
last_updated_friday = today - timedelta(
    days=(weekday - 4) % 7 + (0 if weekday in [2, 3] else 7)
)

# end of previous month with completed updates or end of last year
last_updated_month = (
    last_updated_friday - timedelta(days=last_updated_friday.day)
    if time_filter == "last_12_months"
    else date(today.year - 1, 12, 31)
)
# 12 last completed months with updates
start_query_date = last_updated_month - relativedelta(months=12) + timedelta(days=1)


In [ ]:
# -- Output --
# NOTE: saves in data_in as considered "raw" data
OUTPUT_DIR        = Path("./data_in/Raster_Datasets/EU_GFM")
OUTPUT_LOG_DIR    = Path("./data_out/logs")
OUTPUT_RESOLUTION = 100      # metres
OUTPUT_NODATA     = 255      # uint8 — valid flood-day counts are 0..~31

# -- STAC --
STAC_API_URL   = "https://stac.eodc.eu/api/v1"
GFM_COLLECTION = "GFM"
GFM_BAND       = "ensemble_flood_extent"

# -- Dask tuning for your laptop --
# (Intel Ultra 9 285H, 32 GB RAM, Docker container)
DASK_N_WORKERS       = 6       # keep headroom for OS + Docker overhead
DASK_THREADS_PER_W   = 2       # 2 threads per worker → 12 threads total
DASK_MEMORY_LIMIT    = "4GiB"  # per worker → ~24 GiB ceiling
DASK_CHUNK_XY        = 4096    # spatial chunk size in pixels (at 100m)

# Adaptive time chunk: input (uint8) + Peak
# Peak ≈ Overhead for one time-slice (comparison + sum)
BYTES_PER_PIXEL = 1     # input dtype
PEAK_OVERHEAD = 4       # roughly 2/4× during so-called reduction
TARGET_TASK_MIB = 500   # target each task ≈ 500 MiB


##### 2.  HELPERS

In [ ]:
def init_run_metadata() -> dict:
    """
    Initialise the run metadata dict with parameters,
    environment info, and input configuration.
    Written to JSON at end of run for provenance.
    """
    return {
        "pipeline": "gfm_flood_days",
        "created_at": datetime.now().strftime("%Y%m%dT%H%M%S"),
        # ── Inputs ──
        "iso3": c_iso3,
        "boundary_source": geojson_file[0],
        "window_start": start_query_date,
        "window_end": last_updated_month,
        "lookback_months": time_filter,
        # ── STAC ──
        "stac_api_url": STAC_API_URL,
        "stac_collection": GFM_COLLECTION,
        "stac_asset": GFM_BAND,
        # ── Processing parameters ──
        "output_resolution_m": OUTPUT_RESOLUTION,
        "output_nodata": OUTPUT_NODATA,
        "output_crs": None,         # filled after CRS selection
        "groupby": "solar_day",
        "resampling": "max",
        "output_format": "COG",
        "output_compression": "DEFLATE",
        # ── Dask configuration ──
        "dask": {
            "n_workers": DASK_N_WORKERS,
            "threads_per_worker": DASK_THREADS_PER_W,
            "memory_limit": DASK_MEMORY_LIMIT,
            "chunk_xy": DASK_CHUNK_XY,
            "time_chunk": None,      # filled after adaptive calculation
        },
        # ── Environment ──
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "odc_stac": lib_metadata.version("odc-stac"),
            "dask": lib_metadata.version("dask"),
            "pystac_client": lib_metadata.version("pystac-client"),
            "xarray": lib_metadata.version("xarray"),
            "rioxarray": lib_metadata.version("rioxarray"),
            "rasterio": lib_metadata.version("rasterio"),
            "numpy": lib_metadata.version("numpy"),
        },
        # ── Spatial ──
        "country_bounds_epsg4326": None,    # filled after boundary load
        "grid_shape_px": None,              # filled after CRS/resolution
        "n_spatial_chunks": None,
        # ── Per-month results ──
        "months": {},
        # ── Grid consistency tracking ──
        "grid_shapes_by_month": {},
        # ── Summary ──
        "total_items_processed": 0,
        "total_solar_days_processed": 0,
        "total_runtime_s": None,
        "completed_at": None,
    }


In [ ]:
def save_run_metadata(meta: dict, output_dir: Path):
    """Write run metadata JSON alongside the output GeoTIFFs."""
    path = output_dir / f"{meta['run_id']}.json"
    path.write_text(json.dumps(meta, indent=4, default=str))
    log.info(f"  Wrote run metadata → {path}")


In [ ]:
# Get UTM from gdf country centroid (useful as resolution expressed in metres)
def auto_utm_crs(gdf: gpd.GeoDataFrame) -> str:
    """
    Pick a UTM EPSG code from the centroid of a GeoDataFrame.
    gdf assumed as country subnational shapes in EPSG:4326.
    """
    centroid = gdf.geometry.union_all(
        # default option: robust for overlapping, slower for larger geometry numbers
        method="unary"
    ).centroid
    zone_number = int((centroid.x + 180) / 6) + 1
    epsg = 32600 + zone_number if centroid.y >= 0 else 32700 + zone_number
    log.info(f"Auto-selected CRS: EPSG:{epsg}  (UTM zone {zone_number})")
    return f"EPSG:{epsg}"


In [ ]:
# Iterate calendar months in the time window datetime range
def month_range(start: date, end: date):
    """Yield (month_start, month_end) tuples for each calendar month."""
    cursor = start
    while cursor < end:
        next_month = cursor + relativedelta(months=1)
        # month_end is last day of month
        yield cursor, next_month - timedelta(days=1)
        cursor = next_month


In [ ]:
def compute_month_qc(
        flood_days,
        flood_nodata,
        country_proj,
        n_days,
    ):
    """
    Compute QC statistics by rasterizing the country boundary onto the
    month's specific output grid, discriminating three pixel classes:
        - outside country  (from rio.clip)
        - inside country, no satellite coverage  (nodata from source)
        - inside country, observed  (value: value 0..254)
 
    Returns a dict of QC metrics.
    """
    ny_out, nx_out = flood_days.shape
    transform = flood_days.rio.transform()

    # Rasterize country polygon onto this month's exact grid
    country_mask = rasterize(
        [(geom, 1) for geom in country_proj.geometry],
        out_shape=(ny_out, nx_out),
        transform=transform,
        fill=0,
        dtype="uint8",
        all_touched=True,
    )

    flood_values = flood_days.values
    inside_country = country_mask == 1
    is_nodata = flood_values == flood_nodata
    observed = inside_country & ~is_nodata
    flooded = observed & (flood_values > 0)

    n_inside = int(inside_country.sum())
    n_observed = int(observed.sum())
    n_no_coverage = int((inside_country & is_nodata).sum())
    n_flooded = int(flooded.sum())
    max_flood_days = int(flood_values[observed].max()) if n_observed > 0 else 0
 
    coverage_pct = round((n_observed / n_inside) * 100, 1) if n_inside > 0 else 0.0
    pct_flooded = round((n_flooded / n_observed) * 100, 4) if n_observed > 0 else 0.0
 
    log.info(f"  Country pixels: {n_inside:,}")
    log.info(f"  Observed (>=1 pass): {n_observed:,} ({coverage_pct}%)")
    log.info(f"  No satellite coverage: {n_no_coverage:,} ({100 - coverage_pct:.2f}%)")
    log.info(f"  Flooded: {n_flooded:,} ({pct_flooded}% of observed)")
    log.info(f"  Max flood days: {max_flood_days} / {n_days}")
 
    return {
        "output_shape": [ny_out, nx_out],
        "n_inside_country": n_inside,
        "n_observed": n_observed,
        "n_no_coverage": n_no_coverage,
        "coverage_pct": coverage_pct,
        "n_flooded": n_flooded,
        "pct_flooded": pct_flooded,
        "max_flood_days_observed": max_flood_days,
    }


In [ ]:
def query_stac_items(eodc_catalog, bbox_4326, dt_tuple):
    """Search GFM STAC collection and return list of items."""
    search = eodc_catalog.search(
        # collections to search within
        collections=[GFM_COLLECTION],
        # bound box to intersect
        bbox=bbox_4326,
        # time range search (properly expanded to datetime by pystac_client)
        datetime=f"{dt_tuple[0]}/{dt_tuple[1]}",
        # safety ceiling (prevents runaway memory if unexpectedly large)
        max_items=5000,
    )
    # 2 results iteration methods: items or item_collections
    # NOTE: items iterate over individual results, item_collections over pages
    items = list(search.items())
    log.info(
        f"  STAC query  {dt_tuple[0]} → {dt_tuple[1]} : "
        f"{len(items)} items"
    )
    return items


In [ ]:
def process_gfm_month(
    items: list,
    country_gdf: gpd.GeoDataFrame,
    crs: str,
    bbox_4326: tuple,
    time_chunk: int,
    month_label: str,
    output_path: Path,
):
    """
    Load GFM items for one month, compute flood-day count at 100m,
    clip to country boundary, and save as COG.

    Returns a dict of per-month metadata for the run log.
    """
    month_meta = {
        "month": month_label,
        "status": None,
        "n_items": len(items),
        "n_solar_days": None,
        "items_per_day": None,
        "qc": None,
        "output_file": None,
        "output_size_mb": None,
        "elapsed_s": None,
        "flood_nodata": None,
    }

    if not items:
        log.warning(f"  No items for {month_label} — skipping.")
        month_meta["status"] = "skipped_no_items"
        return month_meta
    
    t0 = time.time()

    # --- a. Load into dask-backed xarray via odc-stac ----------------
    #
    # Key parameters:
    #   groupby="solar_day"  → merges same-day passes (Sentinel-1 revisit
    #                          can produce >1 pass/day)
    #   resolution=100       → odc-stac reprojects from Equi7Grid @20m
    #                          to the target CRS @100m
    #   resampling="max"     → during the 20→100m downscale, ANY flooded
    #                          20m pixel makes the parent 100m cell = 1
    #   chunks                → enables lazy/dask computation
    #   fail_on_error=False  → skip corrupt / unreachable tiles gracefully
    #
    log.info(f"  Loading {len(items)} items with odc-stac …")

    # Xarray Dataset (ds)
    ds = odc_stac.load(
        items,
        bands=[GFM_BAND],
        groupby="solar_day",
        resampling="max",
        crs=crs,
        resolution=OUTPUT_RESOLUTION,
        chunks={"x": DASK_CHUNK_XY, "y": DASK_CHUNK_XY, "time": time_chunk},
        bbox=bbox_4326,               # spatial filter (lon/lat tuple)
        dtype="uint8",
        fail_on_error=False,
    )

    if GFM_BAND not in ds:
        # defensive fallback just in case: shouldn't happen
        log.warning(f"  No {GFM_BAND} for none {month_label} items — skipping.")
        month_meta["status"] = f"skipped_no_{GFM_BAND}"
        return month_meta

    flood = ds[GFM_BAND]
    # Nodata from attributes, if present
    flood_nodata = flood.attrs.get("nodata", None)
    if flood_nodata is None:
        # some data sources missed optional STAC extensions: a nodata value should be used in stac_cfg
        log.warning(f"  `nodata` missed for {GFM_BAND} in {month_label}.")
        # set the default output `nodata` for later data filter, but check for Best Practices
        flood_nodata = OUTPUT_NODATA
    
    n_days = flood.sizes["time"]
    items_per_day = len(items) / max(n_days, 1)
    log.info(f"  {n_days} unique solar-day time slices loaded (lazy).")
    log.info(f"  Items/day: {items_per_day:.1f}")

    month_meta["n_solar_days"] = int(n_days)
    month_meta["items_per_day"] = round(items_per_day, 1)
    month_meta["flood_nodata"] = flood_nodata

    # --- b. Binarise and sum along time → flood day count ------------
    # The ensemble_flood_extent layer is already binary (1 = flood,
    # 0 = no flood).  Values > 1 shouldn't appear, but we threshold
    # defensively.  NaN / nodata cells stay 0 (never observed as flooded).
    # Sum across time → number of unique days the pixel was flagged
    # Cap at 255 (uint8 ceiling — impossible for a single month, but safe)
    flood_days = xr.where(
        flood == flood_nodata, 0, (flood > 0)
    ).astype("uint8").sum(dim="time").clip(
        max=OUTPUT_NODATA - 1
    ).astype("uint8")

    # Copy spatial metadata (CRS, transform) from the source
    flood_days = flood_days.rio.write_crs(crs)
    flood_days = flood_days.rio.write_nodata(OUTPUT_NODATA)

    # --- c. Clip to country boundary ---------------------------------
    # re project into a new GeoDataFrame
    country_proj = country_gdf.to_crs(crs)
    flood_days = flood_days.rio.clip(
        country_proj.geometry,
        all_touched=True,
        drop=True,  # shrink extent to clipped area
    )

    # --- d. Compute (trigger dask) and write COG ---------------------
    log.info(f"  Computing {month_label} flood-day raster …")
    flood_days = flood_days.compute()

    # --- QC stats using per-month country mask ------------------------
    qc = compute_month_qc(flood_days, flood_nodata, country_proj, n_days)
    month_meta["qc"] = qc

    output_path.parent.mkdir(parents=True, exist_ok=True)
    flood_days.rio.to_raster(
        str(output_path),
        driver="COG",
        compress="DEFLATE",
        dtype="uint8",
        overview_resampling="nearest",
        blocksize=512,
    )

    elapsed = time.time() - t0
    size_mb = output_path.stat().st_size / 1e6

    log.info(
        f"  ✓ Saved → {output_path}  "
        f"({size_mb:.1f} MB, {elapsed:.0f}s)"
    )

    month_meta.update({
        "status": "completed",
        "output_file": str(output_path.name),
        "output_size_mb": round(size_mb, 2),
        "elapsed_s": round(elapsed, 1),
    })

    return month_meta


##### 3.  MAIN WORKFLOW

In [ ]:
# ── a. Dask cluster init──────────────────────────────────────────
run_t0 = time.time()

# Init run metadata & add run_id
meta = init_run_metadata()
meta["run_id"] = (
    f"{meta['pipeline']}_{meta['iso3']}_{meta['window_start']}_{meta['window_end']}_{meta['created_at']}"
)

log.info("Starting Dask distributed client …")
client = DaskClient(
    n_workers=DASK_N_WORKERS,
    threads_per_worker=DASK_THREADS_PER_W,
    memory_limit=DASK_MEMORY_LIMIT,
)
log.info(f"  Dashboard: {client.dashboard_link}")

# log workers & mem info
# NOTE: tested (4, 7GiB) or (6, 4GiB), for bigger or smaller countries
log.info(f"  Dask Workers & Memory, respectively: {DASK_N_WORKERS}, {DASK_MEMORY_LIMIT}")


In [ ]:
# ── b. Country box & utm CRS from gdf ────────────────────────────
# Bounding box for STAC queries & loading (EPSG:4326)
bounds = gdf.total_bounds   # minx, miny, maxx, maxy
bbox_4326 = tuple(bounds.tolist())
log.info(f"  BBOX (4326): {bbox_4326}")

# NOTE: gdf assumed in EPSG:4326
utm_crs = auto_utm_crs(gdf)
utm_bounds = gdf.to_crs(utm_crs).total_bounds

meta["output_crs"] = utm_crs
meta["country_bounds_epsg4326"] = {
    "west": bbox_4326[0], "south": bbox_4326[1],
    "east": bbox_4326[2], "north": bbox_4326[3],
}


In [ ]:
# ── c. Open STAC catalog, spatial & time-chunking log  ───────
log.info(f"Connecting to STAC API: {STAC_API_URL}")
catalog = STACClient.open(STAC_API_URL)

# Total grid from utm_bounds
nx = math.ceil((utm_bounds[2] - utm_bounds[0]) / OUTPUT_RESOLUTION)
ny = math.ceil((utm_bounds[3] - utm_bounds[1]) / OUTPUT_RESOLUTION)

# Spatial chunks to process
n_spatial_chunks = (
    math.ceil(nx / DASK_CHUNK_XY) * math.ceil(ny / DASK_CHUNK_XY)
)

log.info(f"  Grid: {nx}×{ny} px → {n_spatial_chunks} spatial chunks")

# ── Graph size estimation: warn if likely >300 MiB ───────────
est_graph_mib = n_spatial_chunks * 12  # ~12 MiB per spatial chunk
if est_graph_mib > 300:
    log.warning(
        f"  Estimated graph: ~{est_graph_mib} MiB — consider "
        f"half-month batching for this country."
    )

# time adaption (use 10 unless spatial chunk reduces)
# NOTE: typically nx & ny >>> spatial_chunks, then
# time adaption depends on spatial_chunk rather than country size
# NOTE: use of all time (-1) blows for bigger countries
# relation/explanation with spatial chunks not yet explored
time_chunk = max(10, int(
    (TARGET_TASK_MIB * 1024**2)
    / (DASK_CHUNK_XY**2 * BYTES_PER_PIXEL * PEAK_OVERHEAD)
))

log.info(f"  Spatial & Time chunks, respectively: {DASK_CHUNK_XY}, {time_chunk}")

meta["grid_shape_px"] = {"x": nx, "y": ny}
meta["n_spatial_chunks"] = n_spatial_chunks
meta["dask"]["time_chunk"] = time_chunk


In [ ]:
# ── d. Process month by month ────────────────────────────────────
log.info(
    f"Processing window (12 months): {start_query_date} → {last_updated_month}"
)

# Track the first month's grid shape as reference for drift detection
reference_shape = None

for m_start, m_end in month_range(start_query_date, last_updated_month):
    month_label = m_start.strftime("%Y-%m")
    log.info(f"\n{'='*60}")
    log.info(f"MONTH: {month_label}")
    log.info(f"{'='*60}")

    output_path = OUTPUT_DIR / (
        f"flood_days_{c_iso3}_{month_label}_EPSG_{utm_crs.removeprefix('EPSG:')}.tif"
    )

    # Skip if month for country previously processed
    if output_path.exists():
        size_mb = output_path.stat().st_size / 1e6
        log.info(f"  Output exists ({size_mb:.1f} MB) — skipping.")
        meta["months"][month_label] = {
            "month": month_label,
            "status": "cached",
            "output_file": str(output_path.name),
            "output_size_mb": round(size_mb, 2),
        }
        continue
    
    items_list = query_stac_items(catalog, bbox_4326, (m_start, m_end))

    month_meta = process_gfm_month(
        items=items_list,
        country_gdf=gdf,
        crs=utm_crs,
        bbox_4326=bbox_4326,
        time_chunk=time_chunk,
        month_label=month_label,
        output_path=output_path,
    )

    # ── Grid drift detection ─────────────────────────────────────
    if month_meta.get("qc"):
        shape = tuple(month_meta["qc"]["output_shape"])
        meta["grid_shapes_by_month"][month_label] = list(shape)

        if reference_shape is None:
            reference_shape = shape
            log.info(f"  Reference grid shape: {shape[0]}x{shape[1]}")
        elif shape != reference_shape:
            dy = shape[0] - reference_shape[0]
            dx = shape[1] - reference_shape[1]
            log.warning(
                f"  Grid drift detected: {shape[0]}x{shape[1]} vs "
                f"reference {reference_shape[0]}x{reference_shape[1]} "
                f"(dy={dy:+d}, dx={dx:+d} pixels)"
            )
            month_meta["grid_drift"] = {"dy": dy, "dx": dx}

    meta["months"][month_label] = month_meta
    # NOTE: total items & solar days count only if processed (not skipped)
    meta["total_items_processed"] += month_meta.get("n_items", 0)
    meta["total_solar_days_processed"] += month_meta.get("n_solar_days", 0) or 0

    # Save metadata incrementally (survives crashes)
    save_run_metadata(meta, OUTPUT_LOG_DIR)


In [ ]:
# ── e. Wrap up ───────────────────────────────────────────────────
total_elapsed = time.time() - run_t0
meta["total_runtime_s"] = round(total_elapsed, 1)
meta["completed_at"] = datetime.now().isoformat() + "Z"

# Grid consistency summary
all_shapes = list(meta["grid_shapes_by_month"].values())
if all_shapes:
    unique_shapes = list(set(tuple(s) for s in all_shapes))
    meta["grid_consistency"] = {
        "all_identical": len(unique_shapes) == 1,
        "unique_shapes": unique_shapes,
        "reference_shape": list(reference_shape) if reference_shape else None,
    }
    if len(unique_shapes) > 1:
        log.warning(f"  Grid shapes varied across months: {unique_shapes}")
    else:
        log.info(f"  Grid shapes consistent: {unique_shapes[0]}")

# Summary stats
completed = [
    m for m in meta["months"].values() if m.get("status") == "completed"
]
cached = [
    m for m in meta["months"].values() if m.get("status") == "cached"
]
meta["summary"] = {
    "months_computed": len(completed),
    "months_cached": len(cached),
    "months_skipped": len(meta["months"]) - len(completed) - len(cached),
    "avg_processed_items_per_month": (
        round(meta["total_items_processed"] / max(len(completed), 1), 0)
        if completed else None
    ),
    "avg_processed_seconds_per_month": (
        round(sum(m["elapsed_s"] for m in completed) / max(len(completed), 1), 1)
        if completed else None
    ),
    "avg_processed_coverage_pct": (
        round(np.mean([m["qc"]["coverage_pct"] for m in completed]), 1)
        if completed else None
    ),
    "avg_processed_pct_flooded": (
        round(np.mean([m["qc"]["pct_flooded"] for m in completed]), 4)
        if completed else None
    ),
    "total_output_mb": round(
        sum(m.get("output_size_mb", 0) for m in meta["months"].values()),
        1,
    ),
}

save_run_metadata(meta, OUTPUT_LOG_DIR)
log.info(
    f"\nDone in {total_elapsed:.0f}s.  "
    f"{len(completed)} computed, {len(cached)} cached."
)
log.info(
    f"Outputs in: {OUTPUT_DIR.resolve()}"
)
log.info(f"Run metadata in: {OUTPUT_LOG_DIR.resolve()}")

client.close()
